# Phase 6 · Unity Catalog — governance thật, và một giới hạn thật

Roadmap hứa ba thứ Hive Metastore (Phase 5) không có: catalog **ba tầng** thật
(`catalog.schema.table`, không phải một catalog ngầm định duy nhất), **phân
quyền** thật (GRANT/REVOKE có tác dụng), và **lineage**. Phase này dựng Unity
Catalog OSS để lấy hai thứ đầu — và để thấy tận mắt UC OSS KHÔNG cho thứ thứ ba,
cùng một giới hạn thật không có trong tài liệu quảng cáo.

**Kiến trúc thật của phase này, không phải kiến trúc dự định ban đầu:** kế
hoạch đầu tiên là tháo hẳn Hive Metastore, để Spark/dbt đọc-ghi thẳng qua Unity
Catalog. Thử thật thì vỡ ngay ở bước đầu tiên — `CREATE TABLE` qua Unity
Catalog cần nó tự "vend" credential S3 tạm thời, và bản UC OSS phát hành chính
thức chưa nói chuyện được với MinIO trong luồng đó. Không phải lỗi cấu hình có
thể sửa bằng cách đọc kỹ hơn — đọc thẳng mã nguồn Scala của connector và thử
cả chục cách rồi mới chắc. Quyết định cuối: **giữ Hive Metastore phục vụ dữ
liệu thật** (không đổi từ Phase 5), **Unity Catalog đứng cạnh làm tầng
governance**, nạp bằng cách đăng ký metadata các bảng thật qua REST API —
`scripts/register_unity_catalog.py`, đã chạy trước khi mở notebook này.

Đây không phải cách né lỗi. Đây là bức tranh thật của kỹ sư dữ liệu: ghép nhiều
mã nguồn mở lại với nhau không phải lúc nào cũng khớp hoàn hảo, và biết CHÍNH
XÁC ranh giới đó — cái gì dùng được, cái gì chưa — là một kỹ năng thật, quý hơn
việc thấy mọi thứ chạy trơn tru.

## Chuẩn bị

In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote(os.environ['SPARK_REMOTE']).getOrCreate()

print('Spark Connect  :', os.environ['SPARK_REMOTE'])
print('Catalog mặc định:', spark.catalog.currentCatalog())
print('Unity Catalog  :', os.environ['UNITY_CATALOG_URL'])

Spark Connect  : sc://spark-connect:15002


Catalog mặc định: spark_catalog
Unity Catalog  : http://unity-catalog:8080


---
## Bước 1 · Catalog BA TẦNG — thứ Hive Metastore không có

Hive Metastore (Phase 5) chỉ có đúng MỘT catalog ngầm định — mọi bảng chỉ có
hai phần `schema.table`. Unity Catalog thêm một tầng thật ở trên.

In [2]:
spark.sql('SHOW CATALOGS').show()
print()
spark.sql('SHOW SCHEMAS FROM unity').show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+


+---------+
|namespace|
+---------+
|   bronze|
|     gold|
|   silver|
+---------+



In [3]:
for schema in ['bronze', 'silver', 'gold']:
    print(f'── unity.{schema} ' + '─' * 40)
    spark.sql(f'SHOW TABLES FROM unity.{schema}').show(truncate=False)
    print()

── unity.bronze ────────────────────────────────────────
+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|bronze   |yellow_trips|false      |
+---------+------------+-----------+


── unity.silver ────────────────────────────────────────


+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|silver   |silver_trips|false      |
|silver   |silver_zones|false      |
|silver   |taxi_zones  |false      |
+---------+------------+-----------+


── unity.gold ────────────────────────────────────────


+---------+-----------------------+-----------+
|namespace|tableName              |isTemporary|
+---------+-----------------------+-----------+
|gold     |gold_daily_zone_revenue|false      |
|gold     |gold_hourly_demand     |false      |
+---------+-----------------------+-----------+




Những bảng này không phải Spark tự khám phá ra — `scripts/register_unity_catalog.py`
đã ĐĂNG KÝ từng bảng một qua REST API của Unity Catalog (không qua Spark),
đọc schema thật bằng `DESCRIBE DETAIL` rồi POST lên `/api/2.1/unity-catalog/tables`.
Mở file đó ra đọc — nó ngắn, và là bằng chứng cho toàn bộ câu chuyện phase này:
Unity Catalog biết "bảng nào, cột gì, ở đâu" mà không cần chạm vào một byte dữ
liệu nào.

---
## Bước 2 · Pipeline THẬT vẫn qua Hive Metastore — không đổi từ Phase 5

`spark.catalog.currentCatalog()` ở trên đã in ra `spark_catalog`, không phải
`unity`. dbt, `scripts/ingest_bronze.py`, mọi thứ trong `make dbt` vẫn gọi tên
bảng KHÔNG tiền tố (`silver.silver_trips`) và tự động vào catalog Hive
Metastore — đúng cơ chế Phase 5, không sửa một dòng SQL nào.

In [4]:
bang = ['bronze.yellow_trips', 'silver.silver_trips', 'silver.silver_zones',
        'gold.gold_daily_zone_revenue', 'gold.gold_hourly_demand']
for b in bang:
    n = spark.table(b).count()
    print(f'{b:<32} {n:>12,}')

bronze.yellow_trips                41,169,720


silver.silver_trips                35,613,229


silver.silver_zones                       265


gold.gold_daily_zone_revenue           80,523


gold.gold_hourly_demand                 1,300


Khớp tuyệt đối với Phase 4/5: 41.169.720 → 35.613.229 → 80.523 + 1.300.
Unity Catalog đứng cạnh, không đứng giữa — pipeline dữ liệu không hề biết nó
tồn tại.

---
## Bước 3 · SAI CÓ CHỦ ĐÍCH — metadata thì được, dữ liệu thì không

Bước 1 vừa chứng minh `SHOW SCHEMAS` / `SHOW TABLES` qua `unity.*` chạy tốt.
Thử một bước xa hơn — đọc THẬT một bảng qua Unity Catalog:

In [5]:
try:
    spark.sql('SELECT * FROM unity.gold.gold_daily_zone_revenue LIMIT 5').show()
except Exception as e:
    print(type(e).__name__, '—', str(e)[:400])

UnknownException — (java.nio.file.AccessDeniedException) s3://lakehouse/phase4/gold/gold_daily_zone_revenue/_delta_log: getFileStatus on s3://lakehouse/phase4/gold/gold_daily_zone_revenue/_delta_log: software.amazon.awssdk.services.s3.model.S3Exception: Forbidden (Service: S3, Status Code: 403, Request ID: 18CEF0CA4807A0F7, Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8):null




Lỗi thật, chép nguyên văn từ lần chạy thử của phase này:

```
AccessDeniedException: s3://lakehouse/phase4/gold/gold_daily_zone_revenue/_delta_log:
getFileStatus on s3://.../_delta_log: ... S3Exception: Forbidden (Service: S3,
Status Code: 403 ...)
```

Chuỗi nhân quả thật (đã lần theo từng bước):

1. `SELECT` trên bảng Delta cần đọc `_delta_log/` để biết version hiện tại —
   khác `SHOW TABLES` (chỉ hỏi Unity Catalog "có bảng gì", không đụng file).
2. Đọc file nghĩa là cần credential S3. `unitycatalog-spark` (Spark plugin
   Unity Catalog dùng) KHÔNG cho Spark dùng credential MinIO có sẵn của
   chính nó — nó luôn tự gọi `generateTemporaryPathCredentials` để Unity
   Catalog vend credential tạm thời. Không có cờ nào tắt được — đã đọc thẳng
   `UCSingleCatalog.scala` để chắc.
3. Unity Catalog trả về credential tạm — nhưng bản OSS phát hành chính thức
   (v0.6.0) CHƯA hỗ trợ custom S3 endpoint (như MinIO) trong luồng vend đó.
   Tự tay thử thẳng bằng `aws-cli` với đúng access/secret key của MinIO cộng
   một session token do UC cấp: MinIO từ chối ngay — `InvalidTokenId: The
   security token included in the request is invalid`.
4. Hỗ trợ MinIO cho luồng này CÓ tồn tại — ở một nhánh thử nghiệm của cộng
   đồng UC, chưa merge vào bản chính thức
   ([unitycatalog/unitycatalog#890](https://github.com/unitycatalog/unitycatalog/discussions/890)).

Đây là ranh giới thật của OSS: **Unity Catalog OSS quản được METADATA hoàn
toàn độc lập với engine đọc dữ liệu** — đúng thiết kế, và cũng chính là lý do
governance với data-plane tách rời nhau lại vỡ khi một mảnh ghép (ở đây: hỗ
trợ S3-compatible endpoint) chưa bắt kịp phần còn lại.

---
## Bước 4 · Phân quyền THẬT — GRANT có tác dụng thật

`server.authorization=enable` (mặc định ảnh gốc TẮT) khiến GRANT/REVOKE có ý
nghĩa thật. Tạo một principal "analyst" mới, chỉ cấp quyền trên `gold`:

In [6]:
import json
import urllib.request
import urllib.error

UC = os.environ['UNITY_CATALOG_URL']
ADMIN = os.environ['UC_ADMIN_TOKEN']

def uc(method, path, token, body=None):
    req = urllib.request.Request(
        f'{UC}{path}', method=method,
        data=json.dumps(body).encode() if body is not None else None,
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req) as r:
            return r.status, json.loads(r.read() or b'{}')
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read() or b'{}')

# Tạo user — bỏ qua nếu đã tồn tại từ lần chạy trước
status, body = uc('POST', '/api/2.1/unity-catalog/users', ADMIN,
                   {'name': 'Analyst Demo', 'email': 'analyst@lakehouse.local'})
print('user:', status, body.get('email', body))

# Cấp SELECT + USE SCHEMA trên unity.gold — KHÔNG cấp gì trên silver
status, body = uc('PATCH', '/api/2.1/unity-catalog/permissions/schema/unity.gold', ADMIN,
                   {'changes': [{'principal': 'analyst@lakehouse.local',
                                 'add': ['SELECT', 'USE SCHEMA'], 'remove': []}]})
print('grant gold:', status, body)

user: 500 {'error_code': 'INTERNAL', 'details': [{'reason': 'INTERNAL', '@type': 'google.rpc.ErrorInfo'}], 'message': "Couldn't unwrap service."}
grant gold: 200 {'privilege_assignments': [{'principal': 'analyst@lakehouse.local', 'privileges': ['SELECT', 'USE SCHEMA']}]}


Chú ý dòng `user: 500 ... "Couldn't unwrap service."` — một bug thật khác của bản REST reference server: tạo user qua `POST /users` bằng JSON thuần (`urllib`) luôn lỗi 500, kể cả với một email hoàn toàn mới chưa từng tồn tại (đã tự tay kiểm bằng `curl` để chắc không phải do trùng). CLI `bin/uc user create` gọi ĐÚNG endpoint đó nhưng lại chạy được — khác nhau ở cách đóng gói request, không phải endpoint hỏng hoàn toàn. Vì `analyst@lakehouse.local` đã được tạo sẵn bằng CLI
trước khi notebook này chạy, GRANT ngay dưới vẫn thành công — principal chỉ cần tồn tại string trong bảng grant, không cần một user record "sạch". Một ví dụ nhỏ nữa cho chủ đề xuyên suốt phase này: phần mềm ghép từ nhiều mã nguồn mở luôn có vài góc chưa nhẵn.

`UC_ANALYST_TOKEN` (biến môi trường sẵn có trong container này) là PAT tự ký
sẵn cho `analyst@lakehouse.local` — cùng cơ chế tự ký, không cần một Identity
Provider ngoài nào. **Tự tay làm:** mint một token MỚI cho principal bất kỳ
bằng terminal —

```bash
make uc-token PRINCIPAL=ai-do-cung-duoc@lakehouse.local
```

— rồi thay `UC_ANALYST_TOKEN` bên dưới bằng token vừa tạo để xem với một
principal khác.

In [7]:
ANALYST_TOKEN = os.environ['UC_ANALYST_TOKEN']

status, gold = uc('GET', '/api/2.1/unity-catalog/tables?catalog_name=unity&schema_name=gold', ANALYST_TOKEN)
print('gold (đã cấp quyền)  →', status, '—', [t['name'] for t in gold.get('tables', [])])

status, silver = uc('GET', '/api/2.1/unity-catalog/tables?catalog_name=unity&schema_name=silver', ANALYST_TOKEN)
print('silver (CHƯA cấp)    →', status, '—', [t['name'] for t in silver.get('tables', [])])

gold (đã cấp quyền)  → 200 — ['gold_daily_zone_revenue', 'gold_hourly_demand']
silver (CHƯA cấp)    → 200 — []


Analyst thấy đúng hai bảng trong `gold`, danh sách `silver` rỗng — không phải
lỗi 403 ồn ào, mà là "không có gì để thấy" (đúng triết lý bảo mật: không tiết
lộ sự tồn tại của thứ không được phép xem). Đổi token ở trên thành
`ADMIN` và chạy lại — cả hai schema đều hiện đầy đủ, vì admin không bị giới
hạn quyền nào.

---
## Lineage — thứ Unity Catalog OSS KHÔNG có

Roadmap Phần 1 hứa ba thứ: catalog ba tầng ✅, phân quyền ✅, lineage. Kiểm tra
thẳng: UC OSS server (self-hosted, `unitycatalog/unitycatalog`) không tự động
ghi lại "bảng X được tính từ bảng Y qua câu lệnh gì". Lineage trong tài liệu
Databricks là tính năng của **workspace quản lý** — nó bám vào chính compute
Databricks chạy (notebook, job) để tự ghi lại mỗi lần một câu query đọc bảng A
ghi ra bảng B. Bản OSS độc lập, không có phần compute đó, nên không có lineage
đi kèm.

Đây không phải một thiếu sót cần vá — nó là ranh giới thật giữa "mã nguồn mở
Databricks công bố" và "sản phẩm quản lý Databricks bán". Biết ranh giới đó
đáng giá hơn một dòng trong bảng so sánh: dbt lineage (Phần 7, đã có từ
Phase 4, qua `ref()`) vẫn là nguồn lineage thật duy nhất trong dự án này.

---
## Năm câu phải trả lời được trước khi sang Phase 7

1. Vì sao `SHOW TABLES FROM unity.gold` chạy được mà
   `SELECT * FROM unity.gold.gold_daily_zone_revenue` thì không?
2. `generateTemporaryPathCredentials` là gì, và tại sao không có cách nào tắt
   nó khi dùng `unitycatalog-spark` với bảng EXTERNAL?
3. Vì sao MinIO từ chối token UC vend ra, dù access key/secret key đúng?
4. Phân quyền của Unity Catalog trả lời "không có quyền" bằng cách nào — báo
   lỗi rõ ràng, hay im lặng trả về danh sách rỗng? Khác nhau ở điểm nào về
   bảo mật?
5. Vì sao lineage KHÔNG nằm trong Unity Catalog OSS — nó thuộc về lớp nào của
   kiến trúc Databricks thật?

## Cầu nối sang Phase 7

Lakehouse giờ có đủ compute (Spark), SQL warehouse (Trino), governance (Unity
Catalog) — nhưng mọi thứ vẫn chạy bằng tay: `make ingest`, `make dbt`,
`make uc-register`, gõ từng lệnh một, đúng thứ tự, đúng lúc. Không ai canh giờ
9 giờ tối tự chạy pipeline, không ai chạy lại đúng ngày hôm qua nếu nó lỗi.

Đó là việc của Airflow — Phase 7.